# Implementation Notebook

#### Import Libraries

In [54]:
import pandas as pd
import numpy as np
import sqlite3, os
from datetime import datetime, timezone
from sklearn.feature_extraction.text import TfidfVectorizer

#### Config Decisions made during EDA

In [2]:
M_SMOOTHING = 25
MIN_CO_RATINGS = 5
MIN_MOVIE_RATES = 5
TOP_N_USER_RECS = 20
TOP_N_SIMILAR = 15
W_COLLAB = 0.75
W_CONTENT = 0.25
DB_PATH = "../backend/movies.db"

#### Load & Integrity

In [3]:
ratings = pd.read_csv("/home/bukunmi/code/givemore/data/ratings.csv")
movies = pd.read_csv("/home/bukunmi/code/givemore/data/movies.csv")
# imdbId MUST stay a string: zero-padded ("0114709"); pandas would parse it as int
links = pd.read_csv("/home/bukunmi/code/givemore/data/links.csv", dtype={"imdbId": "string"})

In [4]:
assert set(ratings.columns) == {"userId", "movieId", "rating", "timestamp"}
assert set(movies.columns) == {"movieId", "title", "genres"}
assert set(links.columns) == {"movieId", "imdbId", "tmdbId"}

In [5]:
assert ratings["userId"].nunique() == 610
assert movies["movieId"].nunique() == 9742
assert links["movieId"].nunique() == 9742

In [6]:
assert set(ratings.movieId) - set(movies.movieId) == set()
assert set(movies.movieId) - set(links.movieId) == set()
assert ratings[["userId","movieId"]].duplicated().sum() == 0

#### Clean Movie Metadata

In [7]:
movies["raw_title"] = movies["title"].str.replace(r"\s+", " ", regex=True).str.strip()
movies["Year"] = movies["raw_title"].str.extract(r"\((\d{4})(?:[–-]\d{4})?\)\s*$", expand=False)
movies["title"] = movies["raw_title"].str.replace(r"\s*\(\d{4}(?:[–-]\d{4})?\)\s*$", "", regex=True)

In [8]:
movies.drop(columns=["raw_title"], inplace=True)

In [9]:
assert movies["Year"].isna().sum() == 12

In [10]:
movies["genre_list"] = movies["genres"].str.split("|")
movies["genre_list"] = movies["genre_list"].apply(lambda genres: [g for g in genres if g != "(no genres listed)"])

In [11]:
assert (movies["genres"] == "(no genres listed)").sum() == 34

#### Foundational Statistics

In [12]:
C = ratings["rating"].mean()
movie_stats = ratings.groupby("movieId")["rating"].agg(["count", "mean"])
user_stats = ratings.groupby("userId")["rating"].agg(["count", "mean"])

In [13]:
assert user_stats.shape[0] == 610
assert movie_stats.shape[0] == 9724
assert C == 3.501556983616962

#### Popular Fallback

In [14]:
def weighted_score(v, R, C, m):
    weighted_score = (v / (v + m)) * R + (m / (v + m)) * C
    return weighted_score

In [15]:
weighted_scores = movie_stats.apply(lambda row: weighted_score(row["count"], row["mean"], C, M_SMOOTHING), axis=1).sort_values(ascending=False)

In [16]:
weighted_scores.dtype

dtype('float64')

In [17]:
assert weighted_scores.shape[0] == 9724
assert weighted_scores.index[0] == 318 # Shawshank
assert np.isclose(weighted_scores.iloc[0], 4.361224925702994)

#### Rating matrices

In [18]:
R = ratings.pivot_table(index="userId", columns="movieId", values="rating")

In [19]:
movie_index = R.columns.to_numpy()
user_index  = R.index.to_numpy()

In [20]:
B = R.notna().astype(float)
co_ratings = (B.T @ B).to_numpy().copy()
np.fill_diagonal(co_ratings, 0)

In [21]:
Mc = R.sub(user_stats["mean"], axis=0).fillna(0.0).to_numpy()

In [22]:
assert (R/B).shape == (610, 9724)
assert co_ratings.shape == (9724, 9724)
assert Mc.shape == (610, 9724)
assert (co_ratings >= MIN_CO_RATINGS).sum() // 2 == 1_293_963

#### Collaborative similarity (adjusted cosine + IUF)

In [23]:
N_movies = R.shape[1]
w = np.log(N_movies / user_stats["count"].to_numpy())

In [24]:
W = Mc * np.sqrt(w)[:, None]
cand = movie_stats["count"].reindex(movie_index).to_numpy() >= MIN_MOVIE_RATES
Wc = W[:, cand]
cand_movies = movie_index[cand]

In [25]:
norms = np.linalg.norm(Wc, axis=0)
Wn = Wc / np.where(norms == 0, 1, norms)

In [26]:
collab_sim = Wn.T @ Wn  

In [27]:
np.fill_diagonal(collab_sim, 0.0)
collab_sim[co_ratings[np.ix_(cand, cand)] < MIN_CO_RATINGS] = 0.0

In [28]:
assert (collab_sim != 0).any(axis=1).sum() == 3634

#### Content similarity (TF-IDF over genres + title tokens)

In [29]:
movies_title_genres_list = movies[["title", "genre_list"]].copy()
movies_title_genres_list["title_genres"] = movies_title_genres_list.apply(lambda row: row["title"] + " " + " ".join(row["genre_list"]), axis=1)

In [30]:
tfidf = TfidfVectorizer(min_df=2)

In [31]:
tfX = tfidf.fit_transform(movies_title_genres_list["title_genres"])

In [32]:
n = tfX.shape[0]
content_neighbors = {}
BLOCK = 512
mids = movies["movieId"].to_numpy()
for start in range(0, n, BLOCK):
    sims = (tfX[start:start+BLOCK] @ tfX.T).toarray()
    for r in range(sims.shape[0]):
        i = start + r
        sims[r, i] = 0.0
        k = TOP_N_SIMILAR
        idx = np.argpartition(sims[r], -k)[-k:]
        idx = idx[np.argsort(sims[r][idx])[::-1]]
        content_neighbors[int(mids[i])] = [
            (int(mids[j]), float(sims[r][j])) for j in idx if sims[r][j] > 0
        ]

In [33]:
assert content_neighbors.keys() == set(movies["movieId"])

#### Hybrid blend (conditional) → `movie_similarity`

In [34]:
collab_sim_clip = np.clip(collab_sim, 0, 1)

In [35]:
tfpos = {int(m): k for k, m in enumerate(mids)}
cand_pos = {int(m): k for k, m in enumerate(cand_movies)}

rows = []
for m in mids:
    m = int(m); cf = []; covered = set()
    if m in cand_pos:
        ci  = cand_pos[m]
        nbr = np.where(collab_sim_clip[ci] > 0)[0]
        if len(nbr):
            cvec = (tfX[tfpos[m]] @ tfX[[tfpos[int(cand_movies[cj])] for cj in nbr]].T).toarray().ravel()
            for cj, cs in zip(nbr, cvec):
                jm = int(cand_movies[cj])
                cf.append((jm, W_COLLAB*collab_sim_clip[ci, cj] + W_CONTENT*float(cs)))
                covered.add(jm)
        cf.sort(key=lambda kv: kv[1], reverse=True)
    fill = [(j, cs) for j, cs in content_neighbors[m] if j not in covered]
    for rank, (j, score) in enumerate((cf + fill)[:TOP_N_SIMILAR], 1):
        rows.append((m, rank, j, float(score)))

movie_similarity = pd.DataFrame(rows, columns=["movie_id", "rank", "similar_movie_id", "score"])

In [36]:
assert (movie_similarity["movie_id"] != movie_similarity["similar_movie_id"]).all()
# rank is the authoritative order (NOT score - tier boundary can break global descending)
assert movie_similarity.groupby("movie_id")["rank"].apply(
    lambda r: sorted(r) == list(range(1, len(r) + 1))
).all()
# expected gap: exactly 5 movies (min_df=2 empty-vector films) get no rows
assert movie_similarity["movie_id"].nunique() == 9742 - 5

#### User recommendations → `user_recommendations`

In [37]:
S = Mc[:, cand] @ collab_sim_clip

In [38]:
S[0]

array([ 1.72760248, -0.69983415, -0.81573343, ...,  0.        ,
        0.05513516,  0.03485687], shape=(3650,))

In [50]:
rated_mask = B.to_numpy()[:, cand].astype(bool)
S_masked = np.where(rated_mask, -np.inf, S)

rated_by_user = ratings.groupby("userId")["movieId"].agg(set)

urec = []
pad_users = []
for u in range(S_masked.shape[0]):
    uid, row = int(user_index[u]), S_masked[u]
    k = min(TOP_N_USER_RECS, int((row > 0).sum()))          # only positive evidence counts
    top = np.argpartition(row, -k)[-k:] if k else np.array([], dtype=int)
    top = top[np.argsort(row[top])[::-1]]
    recs = [(int(cand_movies[i]), float(row[i])) for i in top]
    if len(recs) < TOP_N_USER_RECS:                          # pad with popularity
        pad_users.append(uid)
        have = {m for m, _ in recs} | rated_by_user[uid]
        for pm, psc in weighted_scores.items():
            if len(recs) >= TOP_N_USER_RECS:
                break
            if int(pm) not in have:
                recs.append((int(pm), float(psc)))
    urec += [(uid, r, m, sc) for r, (m, sc) in enumerate(recs, 1)]


In [51]:
user_recommendations = pd.DataFrame(urec, columns=["user_id", "rank", "movie_id", "score"])

In [52]:
per_user = user_recommendations.groupby("user_id").size()
assert per_user.shape[0] == 610 and (per_user == TOP_N_USER_RECS).all()
assert sum(m in rated_by_user[u] for u, m in zip(user_recommendations["user_id"], user_recommendations["movie_id"])) == 0
assert pad_users == [53]
assert int(user_recommendations.loc[user_recommendations["user_id"] == 53, "movie_id"].iloc[0]) == 318

#### `ratings_summary` (metadata for `/stats`)

In [49]:
ratings_summary = pd.DataFrame(
    [
        ("dataset_name", "MovieLens Latest Small"),
        ("rating_count", str(len(ratings))),
        ("movie_count", str(movies["movieId"].nunique())),
        ("user_count", str(ratings["userId"].nunique())),
        ("generated_at", datetime.now(timezone.utc).isoformat()),
    ],
    columns=["key", "value"],
)
ratings_summary

,key,value
0,dataset_name,MovieLens Latest Small
1,rating_count,100836
2,movie_count,9742
3,user_count,610
4,generated_at,2026-06-05T16:02:47.359593+00:00


#### Assemble output tables → write SQLite + indexes

In [53]:
movies_table = movies[["movieId", "title", "genres", "Year"]].rename(
    columns={"movieId": "movie_id", "Year": "year"}
)
movies_table["year"] = pd.to_numeric(movies_table["year"]).astype("Int64")

# external ids from links.csv (Q17 proved 1:1 coverage)
links_table = links.rename(columns={"movieId": "movie_id", "imdbId": "imdb_id", "tmdbId": "tmdb_id"})
movies_table = movies_table.merge(links_table, on="movie_id", how="left")
movies_table["tmdb_id"] = movies_table["tmdb_id"].astype("Int64")
assert movies_table["imdb_id"].notna().all()
assert (movies_table["imdb_id"].str.len() == 7).all()
assert movies_table["tmdb_id"].isna().sum() == 8
users_table = user_stats.reset_index().rename(
    columns={"userId": "user_id", "count": "rating_count", "mean": "avg_rating"}
)

top_pop = weighted_scores.head(100)
popular_movies = pd.DataFrame({
    "rank": np.arange(1, len(top_pop) + 1),
    "movie_id": top_pop.index.astype(int),
    "score": top_pop.to_numpy(),
    "rating_count": movie_stats.loc[top_pop.index, "count"].to_numpy(),
    "avg_rating": movie_stats.loc[top_pop.index, "mean"].to_numpy(),
})

assert movies_table.shape[0] == 9742
assert users_table.shape[0] == 610
assert popular_movies.shape[0] == 100

In [55]:
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
conn = sqlite3.connect(DB_PATH)

In [ ]:
movies_table.to_sql("movies", conn, index=False, if_exists="replace")
users_table.to_sql("users", conn, index=False, if_exists="replace")
popular_movies.to_sql("popular_movies", conn, index=False, if_exists="replace")
user_recommendations.to_sql("user_recommendations", conn, index=False, if_exists="replace")
movie_similarity.to_sql("movie_similarity", conn, index=False, if_exists="replace")
ratings_summary.to_sql("ratings_summary", conn, index=False, if_exists="replace")

5

In [57]:
cur = conn.cursor()
cur.execute('CREATE INDEX IF NOT EXISTS idx_user_recommendations_user_id ON user_recommendations(user_id)')
cur.execute('CREATE INDEX IF NOT EXISTS idx_movie_similarity_movie_id ON movie_similarity(movie_id)')
cur.execute('CREATE INDEX IF NOT EXISTS idx_movies_title ON movies(title)')
cur.execute('CREATE INDEX IF NOT EXISTS idx_popular_movies_rank ON popular_movies("rank")')
conn.commit()
conn.close()

print(f"wrote {DB_PATH}: {os.path.getsize(DB_PATH)/1e6:.2f} MB")

wrote ../backend/movies.db: 6.41 MB


#### Final DB sanity block (seed of `validate_db.py`)

In [68]:
conn = sqlite3.connect(DB_PATH)

def q(sql):
    return pd.read_sql_query(sql, conn)

# tables exist
tables = set(q("SELECT name FROM sqlite_master WHERE type='table'")["name"])
assert {"movies", "users", "popular_movies", "user_recommendations", "movie_similarity", "ratings_summary"} <= tables

# referential integrity on the generated DB
assert q("SELECT COUNT(*) n FROM user_recommendations ur LEFT JOIN movies m ON ur.movie_id = m.movie_id WHERE m.movie_id IS NULL")["n"][0] == 0
assert q("SELECT COUNT(*) n FROM movie_similarity ms LEFT JOIN movies m ON ms.similar_movie_id = m.movie_id WHERE m.movie_id IS NULL")["n"][0] == 0

# no self-similarity; ranks unique per group
assert q("SELECT COUNT(*) n FROM movie_similarity WHERE movie_id = similar_movie_id")["n"][0] == 0
assert q('SELECT COUNT(*) n FROM (SELECT movie_id FROM movie_similarity GROUP BY movie_id, "rank" HAVING COUNT(*) > 1)')["n"][0] == 0
assert q('SELECT COUNT(*) n FROM (SELECT user_id FROM user_recommendations GROUP BY user_id, "rank" HAVING COUNT(*) > 1)')["n"][0] == 0

# coverage
assert q("SELECT COUNT(*) n FROM movies")["n"][0] == 9742
assert q("SELECT COUNT(*) n FROM users")["n"][0] == 610
assert q("SELECT COUNT(DISTINCT user_id) n FROM user_recommendations")["n"][0] == 610
assert q("SELECT COUNT(*) n FROM user_recommendations")["n"][0] == 610 * TOP_N_USER_RECS
assert q("SELECT COUNT(DISTINCT movie_id) n FROM movie_similarity")["n"][0] == 9742 - 5 
assert q("SELECT COUNT(*) n FROM popular_movies")["n"][0] == 100

conn.close()
print("all DB validation checks passed")

all DB validation checks passed
